# Практическая работа № 6 — Классификация текстов (уровень B)

**Студент:** Barsbek Konurbaev

Цель: самостоятельно очистить текст, построить TF-IDF-признаки, обучить классификатор и разобрать пять ошибочных предсказаний.

## 1. Загрузка реального текстового корпуса
Используется UCI SMS Spam Collection — реальный корпус SMS с метками `ham` и `spam`. Данные загружаются из официального ZIP-архива UCI; для загрузки нужен интернет.

In [ ]:
import re
import zipfile
import urllib.request
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
with urllib.request.urlopen(url, timeout=60) as response:
    archive_bytes = response.read()

with zipfile.ZipFile(BytesIO(archive_bytes)) as z:
    data_file = next(name for name in z.namelist() if name.endswith("SMSSpamCollection"))
    with z.open(data_file) as f:
        df = pd.read_csv(f, sep="\\t", header=None, names=["label", "text"], encoding="utf-8")

print("Размер:", df.shape)
display(df.head())
print(df["label"].value_counts())

## 2. Очистка и разделение данных
Сначала разделяем тексты, затем обучаем TF-IDF только на train. В этой задаче не удаляем отрицания отдельным списком; используем стандартные английские стоп-слова, но убираем из списка слова, связанные с отрицанием.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].map(clean_text)
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2,
    random_state=42, stratify=df["label"]
)

stop_words = set(ENGLISH_STOP_WORDS) - {"no", "nor", "not", "never"}
vectorizer = TfidfVectorizer(
    stop_words=sorted(stop_words),
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Размер TF-IDF train:", X_train_tfidf.shape)
print("Размер TF-IDF test:", X_test_tfidf.shape)
print("Размер словаря:", len(vectorizer.vocabulary_))

## 3. Обучение классификатора

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
pred = model.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Precision (spam):", round(precision_score(y_test, pred, pos_label="spam", zero_division=0), 4))
print("Recall (spam):", round(recall_score(y_test, pred, pos_label="spam", zero_division=0), 4))
print("F1 (spam):", round(f1_score(y_test, pred, pos_label="spam", zero_division=0), 4))
print(classification_report(y_test, pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=["ham", "spam"], display_labels=["ham", "spam"], cmap="Blues")
plt.title("Матрица ошибок SMS Spam")
plt.tight_layout()
plt.show()

## 4. Анализ пяти ошибок модели

In [ ]:
errors = pd.DataFrame({
    "text": X_test.reset_index(drop=True),
    "true_label": y_test.reset_index(drop=True),
    "predicted_label": pd.Series(pred)
})
errors = errors[errors["true_label"] != errors["predicted_label"]].head(5).copy()

def error_hypothesis(row):
    if row["true_label"] == "spam":
        return "Возможны скрытые рекламные формулировки, необычные слова или мало известных модели токенов."
    return "Возможна рекламоподобная лексика в обычном сообщении либо отсутствие явных спам-маркеров."

errors["hypothesis"] = errors.apply(error_hypothesis, axis=1)
display(errors[["text", "true_label", "predicted_label", "hypothesis"]])
print("Всего ошибок на тестовой выборке:", int((pd.Series(pred) != y_test.reset_index(drop=True)).sum()))

## 5. Вывод
Укажите размер TF-IDF-матрицы, метрики для класса spam и причины пяти ошибок. Ограничение: очистка настроена для английского SMS-корпуса; для русского или кыргызского текста потребуются другие правила нормализации и стоп-слова.